# 03. SmoothQuant: Outlier Migration for Accurate W8A8 Inference

- **Category:** 14_quantisation / 03_llm_ptq_algorithms
- **Lineage & Citations:** Xiao et al. (2023, *SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models*).
- **Purpose:** Migrate activation quantization difficulty to weights via an exact per-channel transform $\mathbf{Y} = (\mathbf{X}\text{diag}(\mathbf{s})^{-1})(\text{diag}(\mathbf{s})\mathbf{W})$, enabling fast, accurate W8A8 Tensor Core matrix multiplication across batched prefill and decoding.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. AWQ: Activation-Aware Weight Quantization](02_awq_activation_aware_weight_quantization.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. LLM.int8() Mixed Precision Outlier Decomposition](04_llm_int8_mixed_precision_outlier_decomposition.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT quantize un-smoothed activations directly to INT8 when outlier channels exceed $10\times$ median magnitude (causes severe underflow in non-outlier channels); do NOT apply SmoothQuant without folding $\mathbf{s}^{-1}$ into LayerNorm (separate runtime vector divisions add unnecessary kernel latency).

## Mathematical Foundations & Outlier Migration Transform

1. **Equivalent Mathematical Transformation**:
   $$\mathbf{Y} = \mathbf{X}\mathbf{W} = (\mathbf{X}\text{diag}(\mathbf{s})^{-1})(\text{diag}(\mathbf{s})\mathbf{W}) = \widehat{\mathbf{X}}\widehat{\mathbf{W}}$$
2. **Migration Scale Formulation**:
   $$s_j = \frac{\max_i (|X_{i, j}|)^\alpha}{\max_k (|W_{j, k}|)^{1 - \alpha}}, \quad \alpha = 0.5 \implies s_j = \sqrt{\frac{\max_i |X_{i, j}|}{\max_k |W_{j, k}|}}$$
3. **W8A8 Quantization**:
   $$\widehat{X}_{\text{int8}} = \text{quant\_per\_token}(\widehat{X}), \quad \widehat{W}_{\text{int8}} = \text{quant\_per\_channel}(\widehat{W})$$
4. **INT8 GEMM Acceleration**:
   $$\mathbf{Y} \approx (\widehat{X}_{\text{int8}} \widehat{W}_{\text{int8}}) \odot (s_X \cdot s_W) \implies 2\times \text{ Tensor Core TFLOPS}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/03_llm_ptq_algorithms/03_smoothquant_and_outlier_migration_w8a8.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def compute_smoothquant_scale(x_calib: torch.Tensor, weight: torch.Tensor, alpha: float = 0.5, eps: float = 1e-5) -> torch.Tensor:
    act_max = torch.clamp(torch.max(torch.abs(x_calib), dim=0).values, min=eps)
    weight_max = torch.clamp(torch.max(torch.abs(weight), dim=1).values, min=eps)
    s = (act_max ** alpha) / (weight_max ** (1.0 - alpha))
    return torch.clamp(s, min=eps)

def quantize_per_token_int8(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scale = torch.max(torch.abs(x), dim=-1, keepdim=True).values / 127.0
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(x / scale), -128.0, 127.0)
    return q, scale

def quantize_per_channel_int8(w: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scale = torch.max(torch.abs(w), dim=0, keepdim=True).values / 127.0
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(w / scale), -128.0, 127.0)
    return q, scale

def w8a8_gemm_unfused(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor | None = None) -> torch.Tensor:
    q_x, s_x = quantize_per_token_int8(x)
    q_w, s_w = quantize_per_channel_int8(weight)
    out = (q_x * s_x) @ (q_w * s_w)
    return out + bias if bias is not None else out

def smoothquant_w8a8_gemm(x: torch.Tensor, weight: torch.Tensor, scale: torch.Tensor, bias: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    x_smooth = x / scale.unsqueeze(0)
    w_smooth = weight * scale.unsqueeze(1)
    q_x, s_x = quantize_per_token_int8(x_smooth)
    q_w, s_w = quantize_per_channel_int8(w_smooth)
    out = (q_x * s_x) @ (q_w * s_w)
    if bias is not None: out = out + bias
    return out, x_smooth, w_smooth

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. Smooth scale computation parity
scale = compute_smoothquant_scale(weights["x_calib"], weights["w_orig"], alpha=0.5)
assert float(torch.max(torch.abs(scale - weights["scale"]))) < 1e-5

# 3. Unsmoothed W8A8 output parity
out_unsmoothed = w8a8_gemm_unfused(weights["x_calib"][:8], weights["w_orig"], weights["b_orig"])
assert float(torch.max(torch.abs(out_unsmoothed - weights["out_unsmoothed"]))) < 1e-5

# 4. SmoothQuant W8A8 output and intermediate states
out_smooth, x_smooth, w_smooth = smoothquant_w8a8_gemm(weights["x_calib"][:8], weights["w_orig"], scale, weights["b_orig"])
assert float(torch.max(torch.abs(x_smooth - weights["x_smooth"]))) < 1e-5
assert float(torch.max(torch.abs(w_smooth - weights["w_smooth"]))) < 1e-5
assert float(torch.max(torch.abs(out_smooth - weights["out_smooth"]))) < 1e-5
print("All SmoothQuant parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: Unsmoothed W8A8 vs SmoothQuant W8A8
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    mse = float(torch.mean((y_fp32 - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_unsmoothed = eval_sqnr(weights["out_unsmoothed"])
sqnr_smooth = eval_sqnr(weights["out_smooth"])

print(f"Unsmoothed W8A8 SQNR:  {sqnr_unsmoothed:.2f} dB")
print(f"SmoothQuant W8A8 SQNR: {sqnr_smooth:.2f} dB (gain: +{sqnr_smooth - sqnr_unsmoothed:.2f} dB)")
assert sqnr_smooth > sqnr_unsmoothed + 5.0, "SmoothQuant must achieve >5 dB SQNR gain over naive W8A8!"

In [ ]:
# Latency Microbenchmark: Fused Activation Channel Smoothing
b, d_in = 128, 4096
x_mock = torch.randn(b, d_in)
scale_mock = torch.randn(d_in).abs() + 0.5

t0 = time.perf_counter()
for _ in range(50):
    x_smooth = x_mock / scale_mock
t_smooth = (time.perf_counter() - t0) / 50 * 1e3

print(f"Activation Channel Smoothing (128x4096) latency: {t_smooth:.3f} ms")
assert t_smooth < 5.0, "Activation channel smoothing must execute in <5 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Asymmetric Quantization Difficulty**: Activations have systemic outlier channels while weights do not; SmoothQuant migrates difficulty to weights.
2. **Geometric Balancing ($\alpha = 0.5$)**: Splitting dynamic range equally between activations and weights yields optimal W8A8 accuracy.
3. **Hardware Tensor Core GEMM**: Smoothing enables uniform per-token activation quantization compatible with high-throughput INT8 Tensor Cores.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_smoothquant_and_outlier_migration_w8a8.md](../../../markdown/14_quantisation/03_llm_ptq_algorithms/03_smoothquant_and_outlier_migration_w8a8.md)
- **PyTorch 2.x Module:** [03_smoothquant_and_outlier_migration_w8a8.py](../../../pytorch2/14_quantisation/03_llm_ptq_algorithms/03_smoothquant_and_outlier_migration_w8a8.py)
- **Functional JAX Module:** [03_smoothquant_and_outlier_migration_w8a8.py](../../../jax/14_quantisation/03_llm_ptq_algorithms/03_smoothquant_and_outlier_migration_w8a8.py)
- **Native Rust Candle Track:** [03_smoothquant_and_outlier_migration_w8a8.rs](../../../candle/14_quantisation/03_llm_ptq_algorithms/03_smoothquant_and_outlier_migration_w8a8.rs)
- **Hardware Kernel (CPU / WGSL):** [03_smoothquant_and_outlier_migration_w8a8.rs](../../../kernels/14_quantisation/03_llm_ptq_algorithms/03_smoothquant_and_outlier_migration_w8a8.rs)
- **Golden SafeTensors:** [03_smoothquant_and_outlier_migration_w8a8.safetensors](../../../weights/14_quantisation/03_llm_ptq_algorithms/03_smoothquant_and_outlier_migration_w8a8.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. AWQ: Activation-Aware Weight Quantization](02_awq_activation_aware_weight_quantization.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 04. LLM.int8() Mixed Precision Outlier Decomposition](04_llm_int8_mixed_precision_outlier_decomposition.ipynb) &rarr;